# Datathon VinU - Multi GPU Modeling (Kaggle)

Notebook nay duoc thiet ke de chay tren Kaggle voi 2 GPU T4. Co che multi-GPU: train 2 target song song (`Revenue` tren GPU0, `COGS` tren GPU1).

Output chinh:
- `/kaggle/working/outputs/submission_modeling.csv`
- `/kaggle/working/outputs/metrics_summary.json`
- `/kaggle/working/outputs/oof_revenue.csv`
- `/kaggle/working/outputs/oof_cogs.csv`
- `/kaggle/working/outputs/report_bundle.zip`


In [ ]:
# Kaggle setup (run once/session)
!pip -q install --upgrade xgboost optuna holidays prophet joblib


In [ ]:
import json
import warnings
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple

import holidays
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from joblib import Parallel, delayed
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
SEED = 42
DATE_COL = "date"
LAGS = [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 180, 365]
ROLL_WINDOWS = [7, 14, 28, 56, 90]
EWM_ALPHAS = [0.05, 0.1, 0.2, 0.4]
EVENT_COLS = ["is_tet_week", "is_dd_9_9", "is_dd_10_10", "is_dd_11_11", "is_dd_12_12", "is_black_friday"]

TET_DATES = pd.to_datetime([
    "2012-01-23", "2013-02-10", "2014-01-31", "2015-02-19", "2016-02-08",
    "2017-01-28", "2018-02-16", "2019-02-05", "2020-01-25", "2021-02-12",
    "2022-02-01", "2023-01-22", "2024-02-10", "2025-01-29"
])

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

candidates = [
    Path("/kaggle/input/datathon-vinu/dataset"),
    Path("/kaggle/input/datathon_vinu/dataset"),
    Path("/kaggle/input/datathon-vinu"),
    Path("/kaggle/input/datathon_vinu"),
    Path("/kaggle/working/dataset"),
    Path("./dataset"),
]
DATA_DIR = None
for p in candidates:
    if (p / "sales.csv").exists() and (p / "sample_submission.csv").exists():
        DATA_DIR = p
        break
if DATA_DIR is None:
    raise FileNotFoundError("Khong tim thay sales.csv + sample_submission.csv. Vui long attach dataset vao Kaggle Input.")

gpu_count = torch.cuda.device_count()
print(f"Data dir: {DATA_DIR}")
print(f"Detected CUDA devices: {gpu_count}")
for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
class ExpandingWindowWalkForward:
    def __init__(self, initial_train_days=365*5, val_days=180, step_days=180, gap_days=28, max_splits=8):
        self.initial_train_days = initial_train_days
        self.val_days = val_days
        self.step_days = step_days
        self.gap_days = gap_days
        self.max_splits = max_splits

    def split(self, X: pd.DataFrame) -> Iterator[Tuple[np.ndarray, np.ndarray]]:
        n = len(X)
        idx = np.arange(n)
        train_end = self.initial_train_days
        fold = 0
        while True:
            val_start = train_end + self.gap_days
            val_end = val_start + self.val_days
            if val_end > n or (self.max_splits and fold >= self.max_splits):
                break
            yield idx[:train_end], idx[val_start:val_end]
            train_end += self.step_days
            fold += 1

def add_basic_calendar(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    d = out[DATE_COL].dt
    out["day_of_week"] = d.dayofweek
    out["day_of_month"] = d.day
    out["day_of_year"] = d.dayofyear
    out["week_of_year"] = d.isocalendar().week.astype(int)
    out["month"] = d.month
    out["quarter"] = d.quarter
    out["year"] = d.year
    out["is_weekend"] = (d.dayofweek >= 5).astype("int8")
    out["is_month_start"] = d.is_month_start.astype("int8")
    out["is_month_end"] = d.is_month_end.astype("int8")
    out["is_quarter_start"] = d.is_quarter_start.astype("int8")
    out["is_quarter_end"] = d.is_quarter_end.astype("int8")
    out["time_idx"] = np.arange(len(out), dtype=np.int32)
    return out

def add_cyclical(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col, period in [("day_of_week", 7), ("month", 12), ("day_of_year", 365.25), ("day_of_month", 31)]:
        out[f"{col}_sin"] = np.sin(2*np.pi*out[col]/period)
        out[f"{col}_cos"] = np.cos(2*np.pi*out[col]/period)
    return out

def add_tet_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    d = out[DATE_COL].values.astype("datetime64[D]")
    tet = TET_DATES.values.astype("datetime64[D]")
    nxt = np.clip(np.searchsorted(tet, d, side="left"), 0, len(tet)-1)
    prv = np.clip(nxt-1, 0, len(tet)-1)
    days_to = (tet[nxt] - d).astype(int)
    days_since = (d - tet[prv]).astype(int)
    out["days_to_tet"] = days_to
    out["days_since_tet"] = days_since
    out["is_tet_day"] = (days_to == 0).astype("int8")
    out["is_tet_week"] = ((days_to <= 3) | (days_since <= 3)).astype("int8")
    out["is_pre_tet_14d"] = ((days_to >= 1) & (days_to <= 14)).astype("int8")
    out["is_pre_tet_30d"] = ((days_to >= 1) & (days_to <= 30)).astype("int8")
    out["tet_proximity"] = np.exp(-np.minimum(days_to, days_since)/7.0)
    return out

def _black_friday(year: int) -> pd.Timestamp:
    nov = pd.date_range(f"{year}-11-01", f"{year}-11-30", freq="D")
    return nov[nov.dayofweek == 4][3]

def add_event_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    years = sorted(out[DATE_COL].dt.year.unique().tolist())
    vn_holidays = holidays.country_holidays("VN", years=years)
    out["is_vn_holiday"] = out[DATE_COL].isin(vn_holidays).astype("int8")

    mega = {
        "dd_3_3": (3, 3), "dd_6_6": (6, 6), "dd_9_9": (9, 9),
        "dd_10_10": (10, 10), "dd_11_11": (11, 11), "dd_12_12": (12, 12),
    }
    for name, (m, d) in mega.items():
        out[f"is_{name}"] = ((out[DATE_COL].dt.month == m) & (out[DATE_COL].dt.day == d)).astype("int8")

    bfs = pd.to_datetime([_black_friday(y) for y in range(min(years), max(years)+2)])
    out["is_black_friday"] = out[DATE_COL].isin(bfs).astype("int8")
    out["is_cyber_monday"] = out[DATE_COL].isin(bfs + pd.Timedelta(days=3)).astype("int8")
    return out

def make_lag_roll_features(series: pd.Series, horizon: int = 1) -> pd.DataFrame:
    s = series.astype(float)
    out = pd.DataFrame(index=series.index)
    for lag in LAGS:
        out[f"lag_{lag}"] = s.shift(lag)
    for lag in [7, 14, 21, 28, 35, 49]:
        out[f"dow_lag_{lag}"] = s.shift(lag)

    base = s.shift(horizon)
    for w in ROLL_WINDOWS:
        r = base.rolling(w, min_periods=max(3, w//3))
        out[f"rmean_{w}"] = r.mean()
        out[f"rstd_{w}"] = r.std()
        out[f"rmin_{w}"] = r.min()
        out[f"rmax_{w}"] = r.max()
        out[f"rmed_{w}"] = r.median()
        out[f"rcv_{w}"] = out[f"rstd_{w}"] / (out[f"rmean_{w}"].abs() + 1e-6)

    for a in EWM_ALPHAS:
        e = base.ewm(alpha=a, adjust=False)
        out[f"ewm_a{a}"] = e.mean()
        out[f"ewm_std_a{a}"] = e.std()

    out["diff_1"] = s.shift(1) - s.shift(2)
    out["diff_7"] = s.shift(1) - s.shift(8)
    out["yoy_lag"] = s.shift(365)
    out["yoy_diff"] = s.shift(1) - s.shift(365)
    out["yoy_ratio"] = s.shift(1) / (s.shift(365) + 1.0)
    out["yoy_roll28"] = s.shift(365).rolling(28, min_periods=10).mean()
    return out

def lag_roll_row_from_history(history: pd.Series) -> Dict[str, float]:
    h = history.astype(float).dropna()
    arr = h.to_numpy()
    n = len(arr)
    f = {}
    for lag in LAGS:
        f[f"lag_{lag}"] = float(arr[-lag]) if n >= lag else np.nan
    for lag in [7, 14, 21, 28, 35, 49]:
        f[f"dow_lag_{lag}"] = float(arr[-lag]) if n >= lag else np.nan
    for w in ROLL_WINDOWS:
        m = max(3, w//3)
        if n >= m:
            x = arr[-w:]
            f[f"rmean_{w}"] = float(np.nanmean(x))
            f[f"rstd_{w}"] = float(np.nanstd(x, ddof=1)) if len(x) > 1 else 0.0
            f[f"rmin_{w}"] = float(np.nanmin(x))
            f[f"rmax_{w}"] = float(np.nanmax(x))
            f[f"rmed_{w}"] = float(np.nanmedian(x))
            f[f"rcv_{w}"] = f[f"rstd_{w}"] / (abs(f[f"rmean_{w}"]) + 1e-6)
        else:
            f[f"rmean_{w}"] = np.nan
            f[f"rstd_{w}"] = np.nan
            f[f"rmin_{w}"] = np.nan
            f[f"rmax_{w}"] = np.nan
            f[f"rmed_{w}"] = np.nan
            f[f"rcv_{w}"] = np.nan
    hs = h.copy()
    for a in EWM_ALPHAS:
        f[f"ewm_a{a}"] = float(hs.ewm(alpha=a, adjust=False).mean().iloc[-1]) if n else np.nan
        f[f"ewm_std_a{a}"] = float(hs.ewm(alpha=a, adjust=False).std().iloc[-1]) if n > 1 else 0.0
    f["diff_1"] = float(arr[-1] - arr[-2]) if n >= 2 else np.nan
    f["diff_7"] = float(arr[-1] - arr[-8]) if n >= 8 else np.nan
    f["yoy_lag"] = float(arr[-365]) if n >= 365 else np.nan
    f["yoy_diff"] = float(arr[-1] - arr[-365]) if n >= 365 else np.nan
    f["yoy_ratio"] = float(arr[-1] / (arr[-365] + 1.0)) if n >= 365 else np.nan
    f["yoy_roll28"] = float(np.nanmean(arr[-(365+28):-365])) if n >= (365+10) else np.nan
    return f

def evaluate(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

def fit_blend_weights_grid(oof: pd.DataFrame, lam: float = 0.01):
    u = oof.dropna(subset=["pred_l1", "pred_tweedie", "pred_prophet"])
    y = u["actual"].to_numpy()
    p = u[["pred_l1", "pred_tweedie", "pred_prophet"]].to_numpy()
    best_w = np.array([1.0, 0.0, 0.0])
    best_loss = 1e99
    for w0 in np.arange(0, 1.01, 0.01):
        for w1 in np.arange(0, 1.01, 0.01):
            w2 = 1.0 - w0 - w1
            if w2 < 0:
                continue
            w = np.array([w0, w1, w2])
            yhat = p @ w
            rmse = np.sqrt(np.mean((y - yhat)**2))
            cur = rmse + lam * np.sum(w**2)
            if cur < best_loss:
                best_loss = cur
                best_w = w
    return best_w, float(best_loss)

def train_prophet_safe(train_dates, train_y, pred_dates, seed=42):
    try:
        m = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode="multiplicative",
            changepoint_prior_scale=0.05,
            uncertainty_samples=0,
        )
        m.add_country_holidays(country_name="VN")
        m.fit(pd.DataFrame({"ds": train_dates.values, "y": train_y}), seed=seed)
        fcst = m.predict(pd.DataFrame({"ds": pred_dates.values}))
        return np.clip(fcst["yhat"].to_numpy(), 0.0, None)
    except Exception:
        hist = pd.Series(train_y, index=pd.DatetimeIndex(train_dates.values))
        out = []
        for dt in pd.DatetimeIndex(pred_dates.values):
            ref = dt - pd.Timedelta(days=365)
            if ref in hist.index:
                pred = float(hist.loc[ref])
            else:
                pred = float(hist.iloc[-7:].mean())
            out.append(max(pred, 0.0))
            hist.loc[dt] = pred
        return np.array(out)


In [ ]:
sales = pd.read_csv(DATA_DIR / "sales.csv")
sales.columns = [c.lower() for c in sales.columns]
sales[DATE_COL] = pd.to_datetime(sales[DATE_COL])
sales = sales.sort_values(DATE_COL).reset_index(drop=True)

sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
sub_date_col = "Date" if "Date" in sample_sub.columns else "date"
horizon_dates = pd.DatetimeIndex(pd.to_datetime(sample_sub[sub_date_col]).sort_values().unique())

start_date = pd.Timestamp(sales[DATE_COL].min())
end_date = pd.Timestamp(horizon_dates.max())
det = pd.DataFrame({DATE_COL: pd.date_range(start_date, end_date, freq="D")})
det = add_event_features(add_tet_features(add_cyclical(add_basic_calendar(det))))
det = det.set_index(DATE_COL)

print("Sales rows:", len(sales))
print("Train range:", sales[DATE_COL].min().date(), "->", sales[DATE_COL].max().date())
print("Forecast horizon:", horizon_dates.min().date(), "->", horizon_dates.max().date(), f"({len(horizon_dates)} days)")


In [ ]:
def train_target_on_gpu(target_col: str, gpu_id: int, seed: int = 42, max_splits: int = 8):
    frame = sales[[DATE_COL, target_col]].copy()
    target_series = pd.Series(frame[target_col].to_numpy(dtype=float), index=pd.DatetimeIndex(frame[DATE_COL]), name=target_col)

    lag_feats = make_lag_roll_features(target_series, horizon=1)
    train_df = det.loc[target_series.index].copy().join(lag_feats)
    train_df[DATE_COL] = train_df.index
    train_df[target_col] = target_series.values
    train_df = train_df.dropna(subset=["lag_365", "rmean_90", "yoy_lag", "yoy_roll28"]).reset_index(drop=True)

    feature_cols = [c for c in train_df.columns if c not in [DATE_COL, target_col]]
    train_df = train_df.replace([np.inf, -np.inf], np.nan)
    med = train_df[feature_cols].median()
    train_df[feature_cols] = train_df[feature_cols].fillna(med)

    splitter = ExpandingWindowWalkForward(initial_train_days=365*5, val_days=180, step_days=180, gap_days=28, max_splits=max_splits)
    oof = train_df[[DATE_COL, target_col] + EVENT_COLS].copy().rename(columns={target_col: "actual"})
    oof["pred_l1"] = np.nan
    oof["pred_tweedie"] = np.nan
    oof["pred_prophet"] = np.nan

    for tr_idx, va_idx in splitter.split(train_df):
        X_tr = train_df.loc[tr_idx, feature_cols]
        X_va = train_df.loc[va_idx, feature_cols]
        y_tr = train_df.loc[tr_idx, target_col].to_numpy()
        y_va = train_df.loc[va_idx, target_col].to_numpy()

        l1 = xgb.XGBRegressor(
            n_estimators=3000,
            learning_rate=0.03,
            max_depth=8,
            min_child_weight=20,
            subsample=0.8,
            colsample_bytree=0.9,
            reg_alpha=0.01,
            reg_lambda=0.1,
            objective="reg:squarederror",
            eval_metric="mae",
            tree_method="hist",
            device=f"cuda:{gpu_id}",
            random_state=seed,
        )
        l1.fit(X_tr, np.log1p(np.clip(y_tr, 0, None)), eval_set=[(X_va, np.log1p(np.clip(y_va, 0, None)))], verbose=False)
        pred_l1 = np.expm1(l1.predict(X_va))
        oof.loc[va_idx, "pred_l1"] = np.clip(pred_l1, 0.0, None)

        tw = xgb.XGBRegressor(
            n_estimators=3000,
            learning_rate=0.03,
            max_depth=8,
            min_child_weight=20,
            subsample=0.8,
            colsample_bytree=0.9,
            reg_alpha=0.01,
            reg_lambda=0.1,
            objective="reg:tweedie",
            tweedie_variance_power=1.3,
            eval_metric="rmse",
            tree_method="hist",
            device=f"cuda:{gpu_id}",
            random_state=seed + 11,
        )
        tw.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof.loc[va_idx, "pred_tweedie"] = np.clip(tw.predict(X_va), 0.0, None)

        oof.loc[va_idx, "pred_prophet"] = train_prophet_safe(
            train_dates=train_df.loc[tr_idx, DATE_COL],
            train_y=y_tr,
            pred_dates=train_df.loc[va_idx, DATE_COL],
            seed=seed,
        )

    w, blend_obj = fit_blend_weights_grid(oof, lam=0.01)
    oof["pred_blend"] = w[0]*oof["pred_l1"] + w[1]*oof["pred_tweedie"] + w[2]*oof["pred_prophet"]

    event_multipliers = {}
    for ev in EVENT_COLS:
        sub = oof[(oof[ev] == 1) & oof["pred_blend"].notna()]
        if len(sub) >= 3 and sub["pred_blend"].sum() > 0:
            ratio = float(sub["actual"].sum() / sub["pred_blend"].sum())
            n = float(len(sub))
            k = 10.0
            event_multipliers[ev] = (n*ratio + k) / (n + k)
        else:
            event_multipliers[ev] = 1.0

    oof["pred_blend_event"] = oof.apply(
        lambda r: float(r["pred_blend"] * np.prod([event_multipliers[e] for e in EVENT_COLS if int(r[e]) == 1])) if pd.notna(r["pred_blend"]) else np.nan,
        axis=1,
    )

    residual_model = None
    try:
        rs = oof.dropna(subset=["pred_blend_event"]).copy()
        rs["resid"] = rs["actual"] - rs["pred_blend_event"]
        resid_series = rs.set_index(DATE_COL)["resid"].asfreq("D").fillna(0.0)
        lb = acorr_ljungbox(resid_series, lags=[7, 14, 28], return_df=True)
        if (lb["lb_pvalue"] < 0.05).any() and len(resid_series) > 365:
            residual_model = SARIMAX(
                resid_series,
                order=(1, 0, 1),
                seasonal_order=(1, 0, 1, 7),
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)
    except Exception:
        residual_model = None

    # Final fit on full train
    X_all = train_df[feature_cols]
    y_all = train_df[target_col].to_numpy()

    l1_final = xgb.XGBRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=8,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.9,
        reg_alpha=0.01,
        reg_lambda=0.1,
        objective="reg:squarederror",
        tree_method="hist",
        device=f"cuda:{gpu_id}",
        random_state=seed,
    )
    l1_final.fit(X_all, np.log1p(np.clip(y_all, 0, None)), verbose=False)

    tw_final = xgb.XGBRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=8,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.9,
        reg_alpha=0.01,
        reg_lambda=0.1,
        objective="reg:tweedie",
        tweedie_variance_power=1.3,
        tree_method="hist",
        device=f"cuda:{gpu_id}",
        random_state=seed + 11,
    )
    tw_final.fit(X_all, y_all, verbose=False)

    prophet_full = train_prophet_safe(train_df[DATE_COL], y_all, horizon_dates, seed=seed)
    prophet_full = pd.Series(np.clip(prophet_full, 0.0, None), index=horizon_dates)

    residual_fcst = None
    if residual_model is not None:
        try:
            residual_fcst = pd.Series(residual_model.get_forecast(steps=len(horizon_dates)).predicted_mean.values, index=horizon_dates)
        except Exception:
            residual_fcst = None

    # Recursive forecast
    history = target_series.copy()
    preds = []
    for dt in horizon_dates:
        row = det.loc[dt].to_dict()
        row.update(lag_roll_row_from_history(history))
        X_row = pd.DataFrame([{c: row.get(c, np.nan) for c in feature_cols}]).fillna(med)
        p1 = float(np.expm1(l1_final.predict(X_row)[0]))
        p2 = float(tw_final.predict(X_row)[0])
        p3 = float(prophet_full.loc[dt])
        pred = w[0]*max(p1, 0.0) + w[1]*max(p2, 0.0) + w[2]*max(p3, 0.0)
        for ev in EVENT_COLS:
            if int(det.loc[dt, ev]) == 1:
                pred *= event_multipliers[ev]
        if residual_fcst is not None and dt in residual_fcst.index:
            pred += float(residual_fcst.loc[dt])
        pred = max(float(pred), 0.0)
        preds.append(pred)
        history.loc[dt] = pred

    pred_series = pd.Series(preds, index=horizon_dates)

    eval_rows = oof.dropna(subset=["pred_l1", "pred_tweedie", "pred_prophet", "pred_blend", "pred_blend_event"])
    metrics = {
        "l1": evaluate(eval_rows["actual"].values, eval_rows["pred_l1"].values),
        "tweedie": evaluate(eval_rows["actual"].values, eval_rows["pred_tweedie"].values),
        "prophet": evaluate(eval_rows["actual"].values, eval_rows["pred_prophet"].values),
        "blend": evaluate(eval_rows["actual"].values, eval_rows["pred_blend"].values),
        "blend_event": evaluate(eval_rows["actual"].values, eval_rows["pred_blend_event"].values),
        "blend_objective": float(blend_obj),
    }

    fi = pd.DataFrame({
        "feature": feature_cols,
        "importance_gain_l1": l1_final.feature_importances_,
        "importance_gain_tweedie": tw_final.feature_importances_,
    }).sort_values("importance_gain_l1", ascending=False)

    oof.to_csv(OUTPUT_DIR / f"oof_{target_col}.csv", index=False)
    fi.to_csv(OUTPUT_DIR / f"feature_importance_{target_col}.csv", index=False)

    return {
        "target": target_col,
        "gpu_id": gpu_id,
        "forecast": pred_series,
        "blend_weights": {"l1": float(w[0]), "tweedie": float(w[1]), "prophet": float(w[2])},
        "event_multipliers": event_multipliers,
        "metrics": metrics,
    }


In [ ]:
# Multi-GPU execution: train Revenue va COGS song song tren 2 GPU
targets = ["revenue", "cogs"]
if gpu_count >= 2:
    gpu_ids = [0, 1]
    n_jobs = 2
else:
    gpu_ids = [0, 0]
    n_jobs = 1
    print("Chi phat hien <= 1 GPU; fallback single-process mode.")

jobs = [
    (targets[0], gpu_ids[0], SEED, 8),
    (targets[1], gpu_ids[1], SEED + 1, 8),
]

results = Parallel(n_jobs=n_jobs, backend="loky")(
    delayed(train_target_on_gpu)(target_col=t, gpu_id=g, seed=s, max_splits=k)
    for (t, g, s, k) in jobs
)

result_map = {r["target"]: r for r in results}
submission = pd.DataFrame({
    "Date": horizon_dates.strftime("%Y-%m-%d"),
    "Revenue": np.round(result_map["revenue"]["forecast"].values, 2),
    "COGS": np.round(result_map["cogs"]["forecast"].values, 2),
})
submission.to_csv(OUTPUT_DIR / "submission_modeling.csv", index=False)

summary = {
    "config": {
        "seed": SEED,
        "data_dir": str(DATA_DIR),
        "output_dir": str(OUTPUT_DIR),
        "gpu_count": int(gpu_count),
        "mode": "multi-gpu parallel" if n_jobs == 2 else "single-gpu fallback",
    },
    "revenue": {
        "gpu_id": result_map["revenue"]["gpu_id"],
        "blend_weights": result_map["revenue"]["blend_weights"],
        "event_multipliers": result_map["revenue"]["event_multipliers"],
        "metrics": result_map["revenue"]["metrics"],
    },
    "cogs": {
        "gpu_id": result_map["cogs"]["gpu_id"],
        "blend_weights": result_map["cogs"]["blend_weights"],
        "event_multipliers": result_map["cogs"]["event_multipliers"],
        "metrics": result_map["cogs"]["metrics"],
    },
}
with open(OUTPUT_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_DIR / "submission_modeling.csv")
print("Saved:", OUTPUT_DIR / "metrics_summary.json")
submission.head()


In [ ]:
# Zip cell: dong goi artifact can cho report
report_files = [
    OUTPUT_DIR / "submission_modeling.csv",
    OUTPUT_DIR / "metrics_summary.json",
    OUTPUT_DIR / "oof_revenue.csv",
    OUTPUT_DIR / "oof_cogs.csv",
    OUTPUT_DIR / "feature_importance_revenue.csv",
    OUTPUT_DIR / "feature_importance_cogs.csv",
]

zip_path = OUTPUT_DIR / "report_bundle.zip"
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in report_files:
        if p.exists():
            zf.write(p, arcname=p.name)

print("Report bundle:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / (1024*1024), 3) if zip_path.exists() else 0)
